# Cleaning and validating London Fire Stations location data 

To ensure reproducibility and reduce unnecessary computational overhead, the London Fire Brigade station location data were preprocessed separately and loaded into the coursework notebook in a cleaned, post-processed format. This preprocessing included address cleaning, geocoding, and coordinate validation. Using preprocessed spatial data avoids repeated calls to external geocoding services during notebook execution, reducing the risk of runtime instability caused by API rate limits, connection timeouts, or maximum retry errors. It also improves notebook readability by separating one-time data engineering tasks from the main analytical workflow.

In [5]:
# loading libraries 
# install packages if they are unavailable on your system
import pandas as pd
import geopandas as gpd
#%pip install rapidfuzz
#%pip install osmnx
import osmnx as ox
import numpy as np
import networkx as nx
import re
import seaborn as sns
from geopy.geocoders import Nominatim
import time
from shapely.geometry import Point

LFB_stations = gpd.read_file("Data/LFB_stations.gpkg")
stations_directory = pd.read_csv("Data/Stations_directory (from LFB website 2026).csv") # This is to validate the OSM stations data 

In [7]:
# ------------------- LFB station records -------------------------------

'''
As OSM is community-maintained, the station locations
and names require validation against official London Fire Brigade sources (https://www.london-fire.gov.uk/community/your-borough/)
to ensure completeness and accuracy.
'''

print(f"In LFB stations there are {LFB_stations.shape[0]} rows and {LFB_stations.shape[1]} columns") 
# - there 102 stations in London and one river station (Lambeth) yet 112 in the osm dataset so this will be cleaned and matched with the fire stations directory that I manually created using data from the London Fire Brigade Website

# Check to see if there are any missing values in the name column because it is a unique identfier for each station 

# Check for missing values in name column
print(f"There are {LFB_stations['name'].isna().sum()} missing values in the name column")

# 8 stations with missing name values will be removed as they cannot be validated against the LFB directory
LFB_stations = LFB_stations[LFB_stations['name'].notna()]

LFB_stations["name"].unique()

'''
# There is a mismatch between the names in directory and the names in lfb_stations, 
for instance Feltham Fire Station (G39) → feltham fire station
Mill Hill Fire Stn → mill hill fire station
LFB Woodford → woodford fire station
London Fire Brigade Whitechapel → whitechapel fire station
Deptford → deptford fire station
Biggin Hill → biggin hill fire station

Therefore the names should be standardised
'''

# Also LCY Airfield Fire Station, RAF Northolt Fire Station, Heathrow Airport Fire Service 1 and 2 are not a part of LFB so they will be removed 
non_lfb_stations = 'LCY Airfield|RAF Northolt|Heathrow Airport Fire Service' 

LFB_stations = LFB_stations[~LFB_stations['name'].str.contains(non_lfb_stations, na=False)]
print(f"Stations remaining after removing NA values: {len(LFB_stations)}")

def clean_station_name(name):
    if name is None:
        return None
    name = name.lower().strip()
    # Remove codes in brackets e.g. (G39), (G27)
    name = re.sub(r'\(g\d+\)', '', name)
    # Remove other bracketed content
    name = re.sub(r'\([^)]*\)', '', name)
    # Standardise abbreviations
    name = name.replace(' fire stn', ' fire station')
    name = name.replace('lfb ', '')
    name = name.replace('h28 ', '')
    name = name.replace('london fire brigade ', '')
    # Add 'fire station' if missing
    if 'fire station' not in name:
        name = name + ' fire station'
    # Clean up extra spaces
    name = re.sub(r'\s+', ' ', name).strip()
    return name

LFB_stations['name_clean'] = LFB_stations['name'].apply(clean_station_name)
stations_directory['name_clean'] = stations_directory['name'].apply(clean_station_name)

# Check results
#print(LFB_stations[['name', 'name_clean']].to_string())

# Exact match after cleaning
matched = LFB_stations[LFB_stations['name_clean'].isin(stations_directory['name_clean'])]
unmatched = LFB_stations[~LFB_stations['name_clean'].isin(stations_directory['name_clean'])]

#print(f"Matched stations: {len(matched)}") 94
#print(f"Unmatched stations: {len(unmatched)}") 6
#print(unmatched['name_clean'].head())

'''
stations still unmatched 
beddington & wallington fire station (name error)
lee fire station (name error)
bexleyheath fire station (name error)
putney fire station (wandsworth fire station - which already exists in the data)
woolwich fire station (closed)
heston & isleworth fire station (Heston only)
'''

# manually mapping the correct station

manual_mapping = {
    'beddington & wallington fire station': 'wallington fire station',
    'lee fire station': 'lee green fire station',
    'bexleyheath fire station': 'bexley fire station',
    'heston & isleworth fire station': 'heston fire station',
}

LFB_stations['name_clean'] = LFB_stations['name_clean'].replace(manual_mapping)

LFB_stations_validated = LFB_stations[LFB_stations['name_clean'].isin(stations_directory['name_clean'])]
print(f"Validated stations: {len(LFB_stations_validated)}")

LFB_stations["name_clean"].nunique()

# two missing 
# Which stations are in LFB_stations but not in directory
#in_osm_not_directory = LFB_stations_validated[~LFB_stations_validated['name_clean'].isin(stations_directory['name_clean'])]['name_clean']
#print("In OSM but not directory:")
#print(in_osm_not_directory)

# Which stations are in directory but not in LFB_stations
#in_directory_not_osm = stations_directory[~stations_directory['name_clean'].isin(LFB_stations_validated['name_clean'])]['name_clean']
#print("\nIn directory but not OSM:")
#print(in_directory_not_osm)


geolocator = Nominatim(user_agent="lfb_stations")

missing = stations_directory[stations_directory['name_clean'].isin([
    'dagenham fire station', 'wennington fire station', 
    'enfield fire station', 'forest hill fire station', 
    'wimbledon fire station'
])]

rows = []
for idx, row in missing.iterrows():
    location = geolocator.geocode(row['address'])
    if location:
        rows.append({
            'name': row['name'],
            'name_clean': row['name_clean'],
            'geometry': Point(location.longitude, location.latitude)
        })
        print(f"Found: {row['name']}")
    else:
        print(f"Not found: {row['name']}")
    time.sleep(1)

# Convert rows to GeoDataFrame 
missing_gdf = gpd.GeoDataFrame(rows, crs='EPSG:4326').to_crs('EPSG:27700')# BNG is the default crs for all the datasets

# Then append
LFB_stations_validated = pd.concat([LFB_stations_validated, missing_gdf], ignore_index=True)
print(f"Total stations: {len(LFB_stations_validated)}")

# Only keep columns that will be used for analysis
LFB_stations_validated = LFB_stations_validated[['name_clean', 'geometry']].rename(columns={'name_clean': 'name'})

LFB_stations_validated.to_file(
    "Data/LFB_stations_geocoded.gpkg",
    layer="stations",
    driver="GPKG"
)

# LFB_stations_validated.head()


#To check the stations were correctly located and not just correctly named, I plotted them against a CartoDB basemap (independent from OSM) 
#and visually confirmed the positions looked right. Given the scope and time constraints of this project there was not much more that I could do


# LFB_stations.explore(popup='name', tooltip='name', tiles='CartoDB positron', marker_type='marker')

In LFB stations there are 100 rows and 50 columns
There are 0 missing values in the name column
Stations remaining after removing NA values: 100
Validated stations: 98
Found: Dagenham Fire Station
Found: Wennington Fire Station
Found: Enfield Fire Station
Found: Forest Hill Fire Station
Found: Wimbledon Fire Station
Total stations: 103
